# Finalist pilots to 6,000 updates
Continue two selected 2k checkpoints. Choose names only after reviewing all four pilots. CPU verification first; then bounded GPU training. Save weights, exact predictions and results to Drive. Full training still requires a T4 speed check.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
repo=Path('/content/hse-nlp-finalists')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
subprocess.run([sys.executable,'-m','unittest','homeworks.hw2.tests.test_model_sizes','homeworks.hw2.tests.test_continuation'],check=True)
from homeworks.hw2.finalist_pilots import prepare
selected_names = []  # Set exactly two trial names after reviewing the screen.
prepare(selected_names)


Export and verify prepared/finalist-pilots-v1/ready.json from Drive. Only then select a BF16-capable training GPU with at least 40 GiB memory. Do not Run All.

In [ ]:
from google.colab import drive, runtime
try:
    drive.mount('/content/drive', timeout_ms=90000)
    import os, sys, json, hashlib, tarfile, subprocess, asyncio, signal, time
    from pathlib import Path
    prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/finalist-pilots-v1')
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_verified']
    payload=(prepared/'source.tar').read_bytes()
    assert hashlib.sha256(payload).hexdigest()==ready['source_sha256']
    work=Path('/content/hw2-finalist-pilots')
    work.mkdir(exist_ok=False)
    os.chdir(work)
    Path('source.tar').write_bytes(payload)
    with tarfile.open('source.tar') as archive: archive.extractall('.',filter='data')
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True,timeout=180)
    pilot_log=(prepared/'execution.log').open('w')
    pilot_process=subprocess.Popen([sys.executable,'-u','-m','homeworks.hw2.finalist_pilots','--gpu'],stdout=pilot_log,stderr=subprocess.STDOUT,start_new_session=True)
    async def finish_pilots():
        status={'pid':pilot_process.pid,'started_utc':time.time()}
        try:
            await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=3000)
            status['returncode']=pilot_process.returncode
        except BaseException as error:
            status['error']=type(error).__name__+': '+str(error)
            if pilot_process.poll() is None:
                os.killpg(pilot_process.pid,signal.SIGTERM)
                try: await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=10)
                except asyncio.TimeoutError: os.killpg(pilot_process.pid,signal.SIGKILL)
        finally:
            pilot_log.close()
            status['finished_utc']=time.time()
            try: (prepared/'execution.complete.json').write_text(json.dumps(status,indent=2))
            finally: runtime.unassign()
    pilot_task=asyncio.create_task(finish_pilots())
    print('PILOTS_STARTED',pilot_process.pid,'Automatic notebook-kernel release is armed.')
except BaseException:
    runtime.unassign()
    raise


After compute is released, export predictions and evidence from Drive, verify each manifest and exact token decode, submit each checkpoint once to the course bot, and save each receipt to Drive. Selection uses held-out training questions; bot feedback must not become training targets.